<a href="https://colab.research.google.com/github/rasaltejas/Movie_Recommendation/blob/main/Movies_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Movie Recommendation System**

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [ ]:
movies.head(1)

In [ ]:
credits.head(1)

In [ ]:
movies.info()

In [ ]:
movies = movies.merge(credits, on='title')
movies = movies[['id', 'title', 'cast', 'crew', 'overview', 'keywords', 'genres']]

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
import ast
def convert(obj):
  L = []
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
  L = []
  counter = 0

  for i in ast.literal_eval(obj):
      if counter != 3:
          L.append(i['name'])
          counter += 1
      else:
          break

  return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
def fetch_director(obj):
  L = []
  for i in ast.literal_eval(obj):
    if i['job'] == 'Director':
      L.append(i['name'])
      break
  return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head(1)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ", "") for i in x])

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head()

In [ ]:
new_df = movies[['id', 'title', 'tags']]

In [ ]:
new_df

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df['tags'][0]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [ ]:
new_df.head()

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Calculate the cosine similarity matrix between movie vectors
# This matrix will be used to find similar movies.
similarity = cosine_similarity(vectors)

In [ ]:
similarity[1]

In [ ]:
def recommend(movie):
  # Find the index of the input movie in our filtered DataFrame (new_df)
  # We use the 'movie' parameter directly for lookup and get the first matching index.
  movie_index = new_df[new_df['title'] == movie].index[0]

  # Get the similarity scores for the given movie from the similarity matrix
  distances = similarity[movie_index]

  # Sort movies by their similarity scores in descending order and get the top 5 (excluding itself)
  # enumerate(distances) creates pairs of (index, similarity_score)
  # lambda x:x[1] sorts by the similarity score
  # [1:6] selects the top 5 most similar movies, skipping the first one (which is the movie itself)
  movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6]

  # Print the titles of the recommended movies
  for i in movies_list:
     # i[0] refers to the original index of the movie in the 'new_df' DataFrame
     print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Batman Begins')